# Baseline Evaluation: Product Data Quality Validation System

This notebook demonstrates baseline evaluation of the product data quality validation pipeline using synthetic (or real) mismatches. It covers data loading, pipeline execution, metric computation, and visualization of mismatches.

## 1. Load Synthetic or Real Data

Load a dataset (synthetic or real) into a pandas DataFrame for evaluation. This data should include ground truth labels for mismatches.

In [ ]:
import pandas as pd
import os

# Try both absolute and relative paths for robustness
possible_paths = [
    os.path.join(os.getcwd(), 'product-qc-ai', 'data', 'processed', 'synthetic_quality_control.csv'),
    'product-qc-ai/data/processed/synthetic_quality_control.csv',
    './product-qc-ai/data/processed/synthetic_quality_control.csv',
    '../product-qc-ai/data/processed/synthetic_quality_control.csv'
]

for path in possible_paths:
    if os.path.exists(path):
        data = pd.read_csv(path)
        print(f"Loaded data from: {path}")
        break
else:
    raise FileNotFoundError('synthetic_quality_control.csv not found in any expected location.')

data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'product-qc-ai/data/processed/synthetic_quality_control.csv'

## 2. Run Scoring/Validation Pipeline

Apply the scoring or validation steps of the pipeline to the loaded data to generate predictions or scores. This may involve calling your pipeline functions or querying BigQuery tables.

In [ ]:
# Example: Query mismatch scores from BigQuery (replace with your actual project/dataset/table)
from google.cloud import bigquery

PROJECT_ID = 'your-gcp-project'
DATASET = 'product_qc'
MISMATCH_TABLE = 'mismatch_scores'

client = bigquery.Client(project=PROJECT_ID)
query = f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{MISMATCH_TABLE}`"
mismatch_scores = client.query(query).to_dataframe()
mismatch_scores.head()

## 3. Compute Baseline Metrics

Calculate baseline metrics such as accuracy and coverage using the predictions and ground truth labels.

In [ ]:
# Example: Compute accuracy and coverage
# Assume 'ground_truth_mismatch' column exists in your synthetic data

df = data.merge(mismatch_scores, on='product_id', how='inner')

# Example threshold for flagging a mismatch
df['predicted_mismatch'] = (df['vector_mismatch'] > 0.5) | (df['rule_mismatch'] == True)

accuracy = (df['predicted_mismatch'] == df['ground_truth_mismatch']).mean()
coverage = df['predicted_mismatch'].notnull().mean()

print(f"Accuracy: {accuracy:.2%}")
print(f"Coverage: {coverage:.2%}")

## 4. Display Examples of Mismatches

Identify and display examples where the predictions do not match the ground truth to illustrate mismatches.

In [ ]:
# Display mismatches between predictions and ground truth
import pandas as pd

# Assuming 'results' DataFrame contains columns: 'product_id', 'predicted_flag', 'ground_truth_flag', 'score'
mismatches = results[results['predicted_flag'] != results['ground_truth_flag']]

# Show a few examples
mismatches.head(10)